### Graph IV: Conditional Graph - Practice & Exercises
Welcome to the fourth LangGraph notebook! In this notebook, you will practice implementing conditional routing using conditional edges in LangGraph.

#### Part 1: Follow-Along Practice
##### GoalWe want to build a graph that dynamically routes execution based on the operation symbol (`+` or `-`):
```
       ┌───────────┐
       │   START   │
       └─────┬─────┘
             │
             ▼
       ┌───────────┐
       │  router   │
       └─────┬─────┘
             │
      (Conditional Edge)
       /     │     \
     "+"     │     "-"
    /        │        \
   ▼         │         ▼
┌─────┐      │      ┌──────────┐
│ adder │    │      │subtractor│
└─────┬─┘    │      └───┬──────┘
       │      │          │
       └──────┼──────────┘
              ▼
         ┌───────────┐
         │    END    │
         └───────────┘
```

##### Steps:1. Define the `AgentState` schema containing `number1`, `operation`, `number2`, and `finalNumber`.
2. Create the node functions (`adder`, `subtractor`) and the routing function (`decide_next_node`).
3. Build the graph and connect the conditional edges.
4. Compile and invoke the graph.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# 1. Define the State Schema
class AgentState(TypedDict):
    number1: int
    operation: str
    number2: int
    finalNumber: int

In [ ]:
# 2. Define Nodes and Router
def adder(state: AgentState) -> AgentState:
    state["finalNumber"] = state["number1"] + state["number2"]
    return state

def subtractor(state: AgentState) -> AgentState:
    state["finalNumber"] = state["number1"] - state["number2"]
    return state

def decide_next_node(state: AgentState) -> str:
    if state["operation"] == "+":
        return "addition_path"
    elif state["operation"] == "-":
        return "subtraction_path"

In [ ]:
# 3. Build Graph with Conditional Edges
graph = StateGraph(AgentState)

graph.add_node("adder_node", adder)
graph.add_node("subtractor_node", subtractor)
graph.add_node("router", lambda state: state) # Passthrough

graph.add_edge(START, "router")

graph.add_conditional_edges(
    "router",
    decide_next_node,
    {
        "addition_path": "adder_node",
        "subtraction_path": "subtractor_node"
    }
)

graph.add_edge("adder_node", END)
graph.add_edge("subtractor_node", END)

app = graph.compile()

In [ ]:
# 4. Invoke the Graph
result = app.invoke({"number1": 10, "operation": "-", "number2": 5})
print(result["finalNumber"])

---

#### Part 2: Practice Exercises
##### Exercise 1: Double Routing Calculator
###### GoalBuild a double routing calculator that performs two consecutive calculations dynamically using conditional routing gates.

###### Sample Input```python
{
    "number1": 10,
    "operation": "-",
    "number2": 5,
    "number3": 7,
    "number4": 2,
    "operation2": "+",
    "finalNumber": 0,
    "finalNumber2": 0
}
```

###### Sample Output```python
"Result 1: 5, Result 2: 9"
```

###### Plan1. **Define State Schema:** Create an `AgentState` schema to track `number1`, `operation` (+/-), `number2`, `number3`, `operation2` (+/-), `number4`, `finalNumber` (result of computation 1), and `finalNumber2` (result of computation 2).
2. **Define Node & Router Functions:** Write addition and subtraction nodes for both calculations (e.g. `adder1`, `subtractor1`, `adder2`, `subtractor2`) and two router functions (`route1` and `route2`) to evaluate the operation strings.
3. **Build Graph:** Initialize `StateGraph`, register all mathematical and router nodes, set up conditional edges for the first stage router, connect both first-stage operation nodes to the second router, and set up conditional edges for the second stage router to connect to addition/subtraction 2 (and finally connect them to `END`).
4. **Compile & Invoke:** Compile the graph and run it with the sample input and print both results.

In [ ]:
# TODO: Define your State, Nodes, Routers, Graph, Compile, and Invoke it here


<details>
<summary>💡 Click to see solution</summary>

```python
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

### 1. State Schemaclass AgentState(TypedDict):
    number1: int
    operation: str
    number2: int
    number3: int
    number4: int
    operation2: str
    finalNumber: int
    finalNumber2: int

### 2. Node Functionsdef adder1(state: AgentState) -> AgentState:
    state["finalNumber"] = state["number1"] + state["number2"]
    return state

def subtractor1(state: AgentState) -> AgentState:
    state["finalNumber"] = state["number1"] - state["number2"]
    return state

def adder2(state: AgentState) -> AgentState:
    state["finalNumber2"] = state["number3"] + state["number4"]
    return state

def subtractor2(state: AgentState) -> AgentState: 
    state["finalNumber2"] = state["number3"] - state["number4"]
    return state

### Routing Functionsdef route1(state: AgentState) -> str:
    return "add1" if state["operation"] == "+" else "sub1"

def route2(state: AgentState) -> str:
    return "add2" if state["operation2"] == "+" else "sub2"

### 3. Create Graphgraph = StateGraph(AgentState)

graph.add_node("adder1", adder1)
graph.add_node("subtractor1", subtractor1)
graph.add_node("adder2", adder2)
graph.add_node("subtractor2", subtractor2)
graph.add_node("router1", lambda state: state)
graph.add_node("router2", lambda state: state)

### Connectionsgraph.add_edge(START, "router1")
graph.add_conditional_edges(
    "router1",
    route1,
    {
        "add1": "adder1",
        "sub1": "subtractor1"
    }
)

graph.add_edge("adder1", "router2")
graph.add_edge("subtractor1", "router2")

graph.add_conditional_edges(
    "router2",
    route2,
    {
        "add2": "adder2",
        "sub2": "subtractor2"
    }
)

graph.add_edge("adder2", END)
graph.add_edge("subtractor2", END)

### 4. Compile and Runapp = graph.compile()
output = app.invoke({
    "number1": 10,
    "operation": "-",
    "number2": 5,
    "number3": 7,
    "number4": 2,
    "operation2": "+",
    "finalNumber": 0,
    "finalNumber2": 0
})
print(f"Result 1: {output['finalNumber']}, Result 2: {output['finalNumber2']}")
```
</details>